In [ ]:
from binance.client import Client
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime
import psycopg2
import boto3
import io
import ipywidgets as widgets
%matplotlib inline

In [ ]:
def get_secret(secret_name):
    region_name = "eu-west-1"
    # Create a Secrets Manager client
    client = boto3.client('secretsmanager', region_name='eu-west-1')
    get_secret_value_response = client.get_secret_value(SecretId=secret_name)
    if 'SecretString' in get_secret_value_response:
        secret = get_secret_value_response['SecretString']
    else:
        secret = base64.b64decode(get_secret_value_response['SecretBinary'])
    return json.loads(secret)
def get_current_universe():
    f = io.BytesIO()
    s3 = boto3.client('s3', region_name='eu-west-1')
    s3.download_fileobj('my-binance-config', 'config.json', f)
    f.seek(0)
    res = json.loads(f.getvalue())
    return res['universe']
def download_historical_prices(from_time, univ=None):
    at_time = datetime.datetime.strftime(from_time, '%d-%b-%Y %H:00:00')
    with psycopg2.connect('host=localhost user=postgres password=YOUR_DB_PASSWORD port=5432') as conn:
        conn.autocommit = True
        with conn.cursor() as cur:
            qry = f"select time, symbol, close from market where time >= '{at_time}'"
            if univ:
                qry += f' and symbol in {tuple(univ)}'
            cur.execute(qry)
            data = cur.fetchall()
    data = pd.DataFrame(data, columns=['time', 'symbol', 'close']).set_index('time').sort_index()
    data = data.pivot(columns='symbol').droplevel(0, axis=1)
    return data
def get_trades_on_date(symbol, tgt, _client):
    target_datetime = datetime.datetime(year=tgt.year, month=tgt.month, day=tgt.day)
    trades = []
    all_orders = _client.get_all_orders(symbol=symbol)
    all_orders = sorted(all_orders, key=lambda x: x['time'])
    for item in all_orders:
        timestamp = datetime.datetime.fromtimestamp(int(item['time']/1000))
        if timestamp.date() == tgt:
            if not trades and item['side'] == 'SELL':
                trades.append((target_datetime, 1))
            trades.append((timestamp, 1 if item['side'] == 'BUY' else -1, ))
    if not trades and all_orders and all_orders[-1]['side'] == 'BUY':
        trades.append((target_datetime, 1))
    return pd.DataFrame(trades, columns=['time', symbol]).set_index('time').sort_index()[symbol]
def get_returns_on_trades(symbols, start_time, _live):
    prices = _live.loc[:, symbols.tolist()]
    prices = prices.ffill(axis=0)
    return prices
def plot_asset_returns_day(target_date, _client, _live):
    pos = pd.concat([get_trades_on_date(x, target_date, _client) for x in universe], axis=1)
    pos = pos.replace(np.nan, 0).cumsum(axis=0)
    prices = get_returns_on_trades(pos.columns, pos.index.min(), _live)
    prices = prices.reindex(pos.index, method='ffill')
    returns = prices.diff(axis=0)/prices.shift(1)
    pnl = ((1+returns*pos).cumprod(axis=0)-1).sort_index()
    return pnl
def update_plot(_client, axs, axs_pnl):
    starting = live_cache.index[-1]
    new_prices = download_historical_prices(starting)
    curr_live = pd.concat([live_cache, new_prices], axis=0).drop_duplicates()
    for i in range(5):
        axs[i].clear()
        #axs[i].plot([-1,2],[-2,1])
        target_date = datetime.date.today() - datetime.timedelta(days=i)
        pnl_ts = plot_asset_returns_day(target_date, _client, curr_live)
        pnl = pnl_ts.iloc[-1]
        pnl.plot(kind='bar', ax=axs[i])
        axs[i].set_title(f'{target_date} ({pnl.mean()*100:.2f})')
        axs[i].axhline(y=0, color='black')
        axs[i].set_xlabel('')
        axs[i].set_ylim((-0.1, 0.15))
        axs[i].figure.canvas.draw()
        if i == 0:
            axs_pnl.clear()
            pnl_ts.plot(ax=axs_pnl)
            pnl_ts.mean(axis=1).plot(ax=axs_pnl, label='Portfolio', color='black', linestyle='--')
            axs_pnl.figure.canvas.draw()

In [ ]:
sec = get_secret('binance/binance')
client = Client(sec['api_key'], sec['api_secret'])

In [ ]:
def download_historical_prices(from_time=None, univ=None):
    with psycopg2.connect('host=localhost user=postgres password=YOUR_DB_PASSWORD port=5432') as conn:
        conn.autocommit = True
        with conn.cursor() as cur:
            qry = f"select time, symbol, close from market where 1=1"
            if from_time is not None:
                at_time = datetime.datetime.strftime(from_time, '%d-%b-%Y %H:00:00')
                qry +=  f" and time >= '{at_time}'"
            if univ:
                qry += f' and symbol in {tuple(univ)}'
            print(qry)
            cur.execute(qry)
            data = cur.fetchall()
    data = pd.DataFrame(data, columns=['time', 'symbol', 'close']).set_index('time').sort_index()
    data = data.pivot(columns='symbol').droplevel(0, axis=1)
    conn.close()
    return data
historical_live_base = download_historical_prices(from_time=datetime.datetime.strptime('1-Dec-2020', '%d-%b-%Y'))
historical_live_base

In [ ]:
to_keep_base = [
    "IOTAUSDT",
    "VETUSDT",
    "ICXUSDT",
    "ETCUSDT",
    "BTCUSDT",
    "USDCUSDT",
    "XLMUSDT",
    "EOSUSDT",
    "LTCUSDT",
    "TRXUSDT",
    "XRPUSDT",
    "QTUMUSDT",
    "NEOUSDT",
    "ONTUSDT",
    "NULSUSDT",
    "ADAUSDT",
    "ETHUSDT",
    "LINKUSDT",
    "WAVESUSDT",
    "BTTUSDT",
    "ONGUSDT",
    "HOTUSDT",
    "ZILUSDT",
    "ZRXUSDT",
    "FETUSDT",
    "BATUSDT",
    "XMRUSDT",
    "ZECUSDT",
    "IOSTUSDT",
    "CELRUSDT",
    "DASHUSDT",
    "NANOUSDT",
    "OMGUSDT",
    "THETAUSDT",
    "ENJUSDT",
    "MITHUSDT",
    "MATICUSDT",
    "ATOMUSDT",
    "TFUELUSDT",
    "ONEUSDT",
    "FTMUSDT",
    "ALGOUSDT",
    "GTOUSDT",
    "DOGEUSDT",
    "DUSKUSDT",
    "ANKRUSDT",
    "WINUSDT",
    "COSUSDT",
    "MTLUSDT",
    "TOMOUSDT",
    "COCOSUSDT",
    "PERLUSDT",
    "MFTUSDT",
    "KEYUSDT",
    "FUNUSDT",
    "DOCKUSDT",
    "CVCUSDT",
    "DENTUSDT",
    "WANUSDT",
    "CHZUSDT",
    "BANDUSDT",
    "BEAMUSDT",
    "NPXSUSDT",
    "XTZUSDT",
    "RENUSDT",
    "RVNUSDT",
    "HBARUSDT",
    "NKNUSDT",
    "KAVAUSDT",
    "STXUSDT",
    "ARPAUSDT",
    "RLCUSDT",
    "IOTXUSDT",
    "CTXCUSDT",
    "BCHUSDT",
    "TROYUSDT",
    "VITEUSDT",
    "FTTUSDT",
    "USDSUSDT",
    "OGNUSDT",
    "DREPUSDT",
    "TCTUSDT",
    "WRXUSDT",
    "LSKUSDT",
    "BTSUSDT",
    "BNTUSDT",
    "LTOUSDT",
    "AIONUSDT",
    "MBLUSDT",
    "COTIUSDT",
    "ERDUSDT",
    "STPTUSDT",
    "HCUSDT",
    "DATAUSDT",
    "WTCUSDT",
    "CTSIUSDT",
    "HIVEUSDT",
    "CHRUSDT",
    "GXSUSDT",
    "ARDRUSDT",
    "MDTUSDT",
    "MCOUSDT",
    "STMXUSDT",
    "KNCUSDT",
    "REPUSDT",
    "LRCUSDT",
    "BCHABCUSDT",
    "PNTUSDT",
    "COMPUSDT",
    "ZENUSDT",
    "SCUSDT",
    "SNXUSDT",
    "VTHOUSDT",
    "SXPUSDT",
    "DGBUSDT",
    "MKRUSDT",
    "XZCUSDT",
    "STORJUSDT",
    "DCRUSDT",
    "STORMUSDT",
    "MANAUSDT",
    "YFIUSDT",
    "BLZUSDT",
    "JSTUSDT",
    "BALUSDT",
    "KMDUSDT",
    "IRISUSDT",
    "SOLUSDT",
    "SRMUSDT",
    "ANTUSDT",
    "SANDUSDT",
    "CRVUSDT",
    "DOTUSDT",
    "OCEANUSDT",
    "NMRUSDT",
    "STRATUSDT",
    "LUNAUSDT",
    "RSRUSDT",
    "WNXMUSDT",
    "PAXGUSDT",
    "TRBUSDT",
    "BZRXUSDT",
    "YFIIUSDT",
    "SUSHIUSDT",
    "EGLDUSDT",
    "RUNEUSDT",
    "FIOUSDT",
    "KSMUSDT",
    "DIAUSDT",
    "UMAUSDT",
    "BELUSDT",
    "WINGUSDT",
    "UNIUSDT",
    "OXTUSDT",
    "NBSUSDT",
    "SUNUSDT",
    "AVAXUSDT",
    "HNTUSDT",
    "FLMUSDT",
    "ORNUSDT",
    "UTKUSDT",
    "XVSUSDT",
    "ALPHAUSDT",
    "NEARUSDT",
    "AAVEUSDT",
    "FILUSDT",
    "INJUSDT",
    "AUDIOUSDT",
    "CTKUSDT",
    "AKROUSDT",
    "AXSUSDT",
    "HARDUSDT",
    "DNTUSDT",
    "STRAXUSDT",
    "UNFIUSDT",
    "ROSEUSDT",
    "AVAUSDT",
    "XEMUSDT",
    "SKLUSDT",
    "SUSDUSDT",
    "LENDUSDT",
    "GRTUSDT",
    "JUVUSDT",
    "PSGUSDT",
    "1INCHUSDT",
    "REEFUSDT",
    "ASRUSDT",
    "OGUSDT",
    "ATMUSDT",
    "CELOUSDT",
    "RIFUSDT",
    "BTCSTUSDT",
    "TRUUSDT",
    "BCHSVUSDT",
    "CKBUSDT",
    "TWTUSDT",
    "FIROUSDT",
    "LITUSDT",
    "SFPUSDT",
    "USDSBUSDT",
    "CAKEUSDT",
    "DODOUSDT",
    "ACMUSDT"]
to_keep_trunc = set(['ARDRUSDT', 'WANUSDT', 'ACMUSDT', 'ASRUSDT', 'AVAUSDT', 'DREPUSDT',
       'DCRUSDT', 'OGUSDT', 'WNXMUSDT', 'DUSKUSDT'])
to_keep_down = ['BTCDOWNUSDT', 'ETHDOWNUSDT', 'ADADOWNUSDT', 'LINKDOWNUSDT',
       'BNBDOWNUSDT', 'XTZDOWNUSDT', 'EOSDOWNUSDT', 'TRXDOWNUSDT',
       'XRPDOWNUSDT', 'DOTDOWNUSDT', 'LTCDOWNUSDT', 'UNIDOWNUSDT',
       'SXPDOWNUSDT', 'FILDOWNUSDT', 'YFIDOWNUSDT', 'BCHDOWNUSDT',
       'AAVEDOWNUSDT', 'SUSHIDOWNUSDT', 'XLMDOWNUSDT', '1INCHDOWNUSDT']
to_keep_up = ['1INCHUPUSDT',
 'AAVEUPUSDT',
 'ADAUPUSDT',
 'BCHUPUSDT',
 'BNBUPUSDT',
 'BTCUPUSDT',
 'DOTUPUSDT',
 'EOSUPUSDT',
 'ETHUPUSDT',
 'FILUPUSDT',
 'LINKUPUSDT',
 'LTCUPUSDT',
 'SUSHIUPUSDT',
 'SXPUPUSDT',
 'TRXUPUSDT',
 'UNIUPUSDT',
 'XLMUPUSDT',
 'XRPUPUSDT',
 'XTZUPUSDT',
 'YFIUPUSDT']
to_keep_global = to_keep_base+to_keep_down

In [ ]:
to_keep = historical_live_base.loc['1-Jan-2019':,: ].notna().mean(axis=0).sort_values(ascending=False)
to_keep = to_keep[to_keep>0.1]
prohibited = set(['EURUSDT', 'BUSDUSDT', 'GBPUSDT', 'TUSDUSDT', 'AUDUSDT', 'PAXUSDT', 'PADGUSDT'])
to_keep = [x for x in to_keep.index if 'DOWNUSDT' not in x and 'UPUSDT' not in x and x not in prohibited]
print(len(to_keep))
sorted(to_keep)

In [ ]:
st_time = historical_live_base.index[-1]
st_time = st_time.replace(hour = st_time.hour+1)
new_data = download_historical_prices(st_time, to_keep_global)
historical_live = pd.concat([historical_live_base, new_data], axis=0)
historical_live

In [ ]:
def make_positions_smooth(x, max_active, max_thr, hedge_wt):
    assert max_thr >= 0.
    tmp = x.copy()
    tmp.loc[to_keep_down] = 0.
    #tmp += abs(tmp.min())
    rmidx = tmp.sort_values(ascending=False).iloc[max_active:].index
    tmp.loc[rmidx] = 0.
    tmp.loc[tmp<max_thr] = 0.
    tmp = (tmp>0).astype(int)
    wt_sum = tmp.sum()
    weights = tmp/max_active*(1-hedge_wt)
    weights.loc[to_keep_down] = hedge_wt/len(to_keep_down)
    return weights
def simulate(df, wsize=None, wsize_short=None, max_active=None, max_thr=0.0, hedge_wt=0.0):
    returns = df.diff(1)/df.shift(1)
    returns_adj = returns#/returns.ewm(com=30).std()
    signal = -returns_adj.ewm(com=wsize_short).mean()
    signal[signal.abs() < 1e-10] = 0.
    positions = signal.apply(lambda x: make_positions_smooth(x, max_active, max_thr, hedge_wt), axis=1)
    if False:
        positions_holding = positions.iloc[::3, :].copy()
        positions_holding = positions_holding.reindex(positions.index).ffill()
    else:
        positions_holding = positions
    pnl = (positions_holding.shift(1)*(returns-0.))
    trades = positions_holding.diff(1).abs()
    cost = trades*0.001
    return pnl, positions_holding, cost, signal
def get_prev_date(curr_date, days_before, hours_before):
    res = curr_date - datetime.timedelta(days=days_before, hours=hours_before)
    return res
def plot_dashboard(pnl, pos, cost, signal, start_date=None, do_plot=True):
    if start_date:
        _pnl = pnl.loc[start_date:, :]
        _pos = pos.loc[start_date:, :]
        _cost = cost.loc[start_date:, :]
        _signal = signal.loc[start_date:, :]
    else:
        _pnl = pnl
        _pos = pos
        _cost = cost
        _signal = signal
    (_pnl).sum(axis=1).cumsum().plot()
    tmp = (_pnl - _cost).sum(axis=1)
    tmp.cumsum().plot()
    scores = (tmp.sum(), tmp.mean()/tmp.std(), _cost.sum(axis=1).sum())
    print(scores)
    if do_plot:
        plt.figure()
        monthly = tmp.resample('M').sum()
        monthly.plot(kind='bar', figsize=(10, 5))
        plt.ylabel('Pnl (monthly)')
        print(monthly.iloc[-1])
        plt.figure()
        (_pos>0).sum(axis=1).resample('W').mean().plot()
        plt.ylabel('Num Active (weekly avg)')
        plt.figure()
        weekly = _cost.sum(axis=1).resample('W').sum()
        weekly.plot()
        plt.ylabel('Cost (weekly total)')
    return scores

In [ ]:
p2 = 0
p1 = 25
hedge_wt = 0.2
tod = '1-Jun-2021'
calc_start = datetime.datetime.strptime(tod, '%d-%b-%Y')-datetime.timedelta(hours=24*2)
print(calc_start)
score_params = []
to_keep = [x for x in to_keep_global if x not in to_keep_trunc]
for wsize_short in [10, 15, 20, 25]:
    for p1 in [10, 5, 2]:
        for hedge_wt in [0.]:
            print(wsize_short, p1, hedge_wt)
            pstart = calc_start - datetime.timedelta(days=15)
            prices = historical_live.loc[pstart.strftime('%d-%b-%Y'):, to_keep]
            pnl_global, pos_global, cost_global, signal_global = simulate(prices, None, wsize_short, p1, p2/100, hedge_wt)
            score = plot_dashboard(pnl_global, pos_global, cost_global, signal_global, calc_start.strftime('%d-%b-%Y %H:%M'), False)[0]
            score_params.append((score, (wsize_short, p1, hedge_wt)))
score_params = sorted(score_params, key=lambda x: x[0])
print(score_params[-5:])

In [ ]:
to_keep = [x for x in to_keep_global if x not in to_keep_trunc]
p2 = 0.
wsize_short = 15
hedge_wt = 0.
p1 = 5# best_p1
start_date = '1-Jan-2021'
prices = historical_live.loc[:, to_keep]
pnl_global, pos_global, cost_global, signal_global = simulate(prices, None, wsize_short, p1, p2/100, hedge_wt)
#pnl_global[pnl_global>0.01] = 0.01
#pnl_global[pnl_global<-0.005] = -0.005
plot_dashboard(pnl_global, pos_global, cost_global, signal_global, start_date, False)
mkt_ret = historical_live.loc[:, to_keep_base]
mkt_ret = mkt_ret.diff(1)/mkt_ret.shift(1)
mkt_ret.loc[start_date:, :].mean(axis=1).cumsum().plot(color='k')

In [ ]:
simstart = pnl_global.index[-1].replace(day=1).strftime('%d-%b-%Y')
aa=(pnl_global-cost_global).loc[simstart:, :].sum(axis=1).resample('D').sum()
mkt_ret = historical_live.loc[:, to_keep]
mkt_ret = mkt_ret.diff(1)/mkt_ret.shift(1)
mkt_ret = mkt_ret.loc[simstart:, :]
mkt_ret = mkt_ret.mean(axis=1)
pd.concat([mkt_ret.resample('D').sum(), aa], keys=['Market', 'Trading'], axis=1).plot(kind='bar', figsize=(15,5))
plt.legend(loc='upper left')
plt.ylim((-0.3, 0.5))
plt.grid(True)
plt.figure()
top_k = (pnl_global-cost_global).loc[simstart:, :].sum(axis=0).sort_values(ascending=False)
pd.concat([top_k.iloc[:10], top_k.iloc[-10:]], axis=1, keys=['winners', 'losers']).plot(kind='bar', figsize=(10,5))
plt.figure()
(pnl_global-cost_global).loc[simstart:, :].sum(axis=1).cumsum().plot(marker='.')
(pnl_global-cost_global).loc[simstart:, to_keep_down].sum(axis=1).cumsum().plot(marker='.')
mkt_ret.loc[simstart:].cumsum().plot(color='k', marker='.')
aa.sum()

In [ ]:
start_of_day = (pnl_global.index[-1] - datetime.timedelta(hours=24)).strftime('%d-%b-%Y')
aa=(pnl_global-cost_global).loc[start_of_day:, :]
mkt_ret = historical_live.loc[:, to_keep]
mkt_ret = mkt_ret.diff(1)/mkt_ret.shift(1)
mkt_ret = mkt_ret.loc[start_of_day:, :]
plt.figure()
pd.concat([mkt_ret.mean(axis=1), aa.sum(axis=1)], keys=['Market', 'Trading'], axis=1).plot(kind='bar', figsize=(15,5))
plt.ylim((-.05, 0.075))
plt.grid(True)
plt.legend(loc='lower left')
plt.figure()
top_k = aa.sum(axis=0).sort_values(ascending=False)
pd.concat([top_k.iloc[:10], top_k.iloc[-5:]], axis=1, keys=['winners', 'losers']).plot(kind='bar', figsize=(10,5))
plt.figure()
aa.sum(axis=1).cumsum().plot(marker='.')
aa.loc[:, to_keep_down].sum(axis=1).cumsum().plot(marker='.')
mkt_ret.mean(axis=1).cumsum().plot(color='k', marker='.')
aa.sum().sum()

In [ ]:

start_date = '18-Jun-2021 21:00'
with psycopg2.connect('host=localhost user=postgres password=YOUR_DB_PASSWORD port=5432') as conn:
    with conn.cursor() as cur:
        cur.execute(f"select time, symbol, balance from account where account_name='account1' and time >= '{start_date}'")
        data = cur.fetchall()
conn.close()
balance_split = pd.DataFrame(data, columns=['time', 'symbol', 'balance']).set_index('time')[['balance', 'symbol']]
balance_split = balance_split.pivot(columns='symbol').droplevel(0, axis=1).fillna(0.)
balance = balance_split.sum(axis=1)
balance.plot(marker='o')

In [ ]:
fund_size = 4000
balance_ret = balance.diff(1).shift(-1)/fund_size
sim_ret = (pnl_global-cost_global).sum(axis=1).reindex(balance.index)
pd.concat([balance_ret, sim_ret], axis=1, keys=['real', 'sim']).plot(kind='bar')
plt.grid(True)
plt.figure()
(balance_ret-sim_ret).plot(kind='bar')
plt.grid(True)
plt.figure()
(balance_ret.cumsum()*fund_size+balance.iloc[0]).plot(marker='.')
(sim_ret.cumsum()*fund_size+balance.iloc[0]).plot(marker='.')
plt.figure()
((balance_ret-sim_ret).cumsum()*fund_size).plot()
((balance_ret-sim_ret)*100).expanding().mean().plot(secondary_y=True)
(balance_ret-sim_ret).sum()/(len(sim_ret)-1)*100

In [ ]:
hr = 5
pp=(pos_global).loc[f'17-Jun-2021 {hr}:00']
pp = pp[pp.abs()>0]
print(pp)
xx=(pnl_global-cost_global).loc[f'17-Jun-2021 {hr+1}:00', pp.index]
print(xx)
xx=balance_split.loc[f'17-Jun-2021 {hr+1}:00', [c for c in to_keep_global if 'DOWNUSDT' not in c]]
print(xx[xx>0])
xx=balance_split.loc[f'17-Jun-2021 {hr+2}:00', [c for c in to_keep_global if 'DOWNUSDT' not in c]]
xx[xx>0]

In [ ]:
prev = historical_live.loc[f'17-Jun-2021 {hr}:00', pp.index]
nex = historical_live.loc[f'17-Jun-2021 {hr+1}:00', pp.index]
asset_pnl = (nex-prev)/prev
print(prev)
print(nex)
print(asset_pnl)
print((pnl_global-cost_global).loc[f'17-Jun-2021 {hr+1}:00', pp.index])
print(asset_pnl.mean())
print((pnl_global-cost_global).loc[f'17-Jun-2021 {hr+1}:00', pp.index].sum())

In [ ]:
aa=(pnl_global-cost_global).loc['10-Jun-2021':, :]
dat = []
for idx, row in aa.iterrows():
    curr = (row[row>0]*100).to_numpy().tolist()
    prnt = [x for x in curr if x>1.0]
    if prnt:
        print(idx, prnt)
    dat.extend(curr)
pd.Series(dat).hist(bins=40)

In [ ]:
def dow(dd, bf):
    start = '1-Jan-2000'
    tmp_pnl = pnl_global.loc[start:, :]
    if bf:
        tmp = tmp_pnl.loc[[x for x in tmp_pnl.index if x.day_name() == dd and x.hour < 15]].sum(axis=1)
    else:
        tmp = tmp_pnl.loc[[x for x in tmp_pnl.index if x.day_name() == dd and x.hour >= 15]].sum(axis=1)
    tmp[tmp > tmp.quantile(0.98)] = np.nan
    tmp[tmp < tmp.quantile(0.02)] = np.nan
    plt.figure()
    tmp.hist(bins=40)
    #tmp.plot()
    plt.title(dd)
    return tmp.mean()*100
print(dow('Saturday', True), dow('Saturday', False))
print(dow('Sunday', True), dow('Sunday', False))
print(dow('Monday', True), dow('Monday', False))
print(dow('Tuesday', True), dow('Tuesday', False))
print(dow('Wednesday', True), dow('Wednesday', False))
print(dow('Thursday', True), dow('Thursday', False))
print(dow('Friday', True), dow('Friday', False))

pnl_global.sum(axis=1).mean()

In [ ]:
start_date = datetime.datetime.now() - datetime.timedelta(hours=6)
start_date = start_date.strftime('%d-%b-%Y %H:%M')
with psycopg2.connect('host=localhost user=postgres password=YOUR_DB_PASSWORD port=5432') as conn:
    conn.autocommit = True
    with conn.cursor() as cur:
        qry = f"select time, symbol, (fill_price-target_price)/target_price*100, side from slippage where time >= '{start_date}'"
        cur.execute(qry)
        data = cur.fetchall()
conn.close()
slippage = pd.DataFrame(data, columns=['time', 'symbol', 'slippage', 'side'])
slippage

In [ ]:
tmp = slippage.copy()
tmp['date'] = tmp['time'].apply(lambda x: x.date())
grp = tmp.loc[tmp['side']=='BUY', :].groupby('date').mean()
print(grp.iloc[-3:])
grp.plot(kind='bar', figsize=(15,5))
grp = tmp.loc[tmp['side']=='SELL', :].groupby('date').mean()
grp.plot(ax=plt.gca(), kind='bar', color='g')
plt.axhline(0, color='k')
print(grp.iloc[-3:])

In [ ]:
if False:
    #tmp = slippage.loc[[x in high_vol.index for x in slippage['symbol']], :].copy()
    # tmp = slippage.loc[[x not in truncate_slippage for x in slippage['symbol']], :].copy()
    tmp = slippage.loc[[x not in to_keep_trunc for x in slippage['symbol']], :].copy()
else:
    tmp = slippage.copy()
aa=tmp.loc[slippage['side'] == 'SELL', 'slippage']
aa.hist(bins=40)
plt.title('SELL')
print('SELL', aa.mean())
plt.figure()
aa=tmp.loc[slippage['side'] == 'BUY', 'slippage']
aa.hist(bins=40)
plt.title('BUY')
print('BUY', aa.mean())

In [ ]:
worst_sell=slippage.loc[slippage['side'] == 'SELL', ['symbol', 'slippage']].groupby('symbol')
worst_sell=worst_sell.mean().sort_values('slippage')
worst_sell.plot(kind='bar', figsize=(15, 5))
plt.title('SELL')
plt.figure()
worst_buy=slippage.loc[slippage['side'] == 'BUY', ['symbol', 'slippage']].groupby('symbol')
worst_buy=worst_buy.mean().sort_values('slippage')
worst_buy.plot(kind='bar', figsize=(15,5))
plt.title('BUY')

In [ ]:
start_date = datetime.datetime.now() - datetime.timedelta(hours=24*5)
start_date = start_date.strftime('%d-%b-%Y %H:%M')
with psycopg2.connect('host=localhost user=postgres password=YOUR_DB_PASSWORD port=5432') as conn:
    with conn.cursor() as cur:
        cur.execute(f"select time, symbol, bid, ask from orderbook where time >= '{start_date}'")
        data = cur.fetchall()
conn.close()
spreads = pd.DataFrame(data, columns=['time', 'symbol', 'bid', 'ask'])
spreads['spread'] = (spreads['ask']-spreads['bid'])/spreads['ask']*100
spreads = spreads.drop(['bid', 'ask'], axis=1)
spreads = spreads.pivot(columns='symbol', index='time').droplevel(0, axis=1)
spreads = spreads.loc[:, to_keep_global]
spreads = spreads.loc[: ,[x for x in spreads.columns if 'DOWN' not in x]]
spreads

In [ ]:
toplt = spreads.mean(axis=0).sort_values(ascending=False)[:100].index
spreads.loc[:, toplt].mean(axis=0).plot(kind='bar', figsize=(15,5))
plt.figure()
spreads.mean(axis=0).sort_values(ascending=False).plot()

In [ ]:
traded = set(to_keep_global)
sp = spreads.loc[:, [x for x in spreads.columns if x in traded]].median(axis=0).sort_values(ascending=False).dropna()
truncate_slippage = sp.loc[sp>0.3].index
print(truncate_slippage, len(truncate_slippage))
entry = set(truncate_slippage) - set(to_keep_trunc)
print('Entry', entry)
print(sp.loc[entry])
exit = set(to_keep_trunc) - set(truncate_slippage)
print('Exit', exit)


In [ ]:
[x for x in to_keep_global if x not in truncate_slippage]

In [ ]:
spreads.loc[:, 'LINKUSDT'].plot()

In [ ]:
plt.figure()
axs = plt.gca()
spreads.loc[:, list(truncate_slippage)].rolling(15).mean().plot(color='gray', ax=axs)
plt.ylim((0, 0.7))
plt.legend([])
plt.figure()
spreads.loc[:, [x for x in sp.index if x not in truncate_slippage]].rolling(15).mean().plot(color='gray')
plt.ylim((0, 0.7))
plt.legend([])

In [ ]:
spreads.loc[:, 'KEYUSDT'].rolling(5).mean().plot()
spreads.loc[:, 'DATAUSDT'].rolling(5).mean().plot()

In [ ]:
sp

In [ ]:
from statsmodels.tsa.stattools import pacf
def get_adjusted_pcaf(x, nlags):
    val, cint = pacf(x.dropna(), nlags=nlags, alpha=0.05)
    adj = val[1:]/(cint[1:,1]-cint[1:,0])*2
    return adj
cdata = []
cols =  to_keep_global
start_date = '1-Apr-2021'
for c in cols:
    aa=historical_live.loc[start_date:, c]
    ret = aa.diff(1)/aa.shift(1)
    ret = ret/ret.std()
    ret.loc[ret.abs() > 5] = np.nan
    ret = ret.dropna()
    nlags = 10
    if len(ret) < nlags*2:
        continue
    ser = pd.Series(get_adjusted_pcaf(ret, nlags), index=list(range(1, nlags+1)))
    ser.name = c
    cdata.append(ser)
cdata = pd.concat(cdata, axis=1)
plt.axhspan(-1, 1, alpha=0.5, color='y')
cdata.mean(axis=1).plot(marker='o')


plt.grid(True)
#plt.ylim((-0.2, 0.2))
print(cdata.loc[[1,2,3,4,5], :].mean(axis=1).mean())

In [ ]:
infos

In [ ]:
with psycopg2.connect('host=localhost user=postgres password=YOUR_DB_PASSWORD port=5432') as conn:
    conn.autocommit = True
    with conn.cursor() as cur:
        for info in infos:
            symbol = info['symbol']
            print(symbol)
            for f in info['filters']:
                ftype = f['filterType']
                fval = dict(f)
                del fval['filterType']
                qry = 'insert into symbol_info (symbol, filter_key, filter_value) values (%s, %s, %s)'
                cur.execute(qry, (symbol, ftype, json.dumps(fval)))

In [ ]:
[x for x in infos if x['symbol'] == '1INCHDOWNUSDT']

In [ ]:
from statsmodels.tsa.stattools import pacf
def get_adjusted_pcaf(x, nlags):
    ret = x.diff(1)/x.shift(1)
    ret = ret/ret.std()
    ret.loc[ret.abs() > 5] = np.nan
    ret = ret.dropna()
    if len(ret) < nlags*2:
        return
    val, cint = pacf(ret, nlags=nlags, alpha=0.05)
    adj = val[1:]/(cint[1:,1]-cint[1:,0])*2
    return adj
cdata = []
cols =  perpetual_6h.columns
start_date = '1-Jan-2021'
for c in cols:
    aa=perpetual_6h.loc[start_date:, c]
    nlags = 100
    ser = pd.Series(get_adjusted_pcaf(aa, nlags), index=list(range(1, nlags+1)))
    ser.name = c
    cdata.append(ser)
cdata = pd.concat(cdata, axis=1)
plt.axhspan(-1, 1, alpha=0.5, color='y')
cdata.mean(axis=1).plot(marker='o')
plt.grid(True)

In [ ]:
import time
def get_klines(symbol, starting):
    data = client.get_historical_klines(symbol=symbol, interval=client.KLINE_INTERVAL_1HOUR, start_str=starting)
    ser = pd.Series([vv[4] for vv in data], dtype=float, index=pd.to_datetime([vv[6] for vv in data], unit='ms'))
    ser.name = symbol
    return ser
def get_vol_klines(symbol, starting):
    data = client.get_historical_klines(symbol=symbol, interval=client.KLINE_INTERVAL_1HOUR, start_str=starting)
    ser = pd.Series([vv[7] for vv in data], dtype=float, index=pd.to_datetime([vv[6] for vv in data], unit='ms'))
    ser.name = symbol
    return ser
symbols = to_keep_global
starting = int(datetime.datetime.strptime('1-Jun-2021', '%d-%b-%Y').timestamp()*1000)
cols = []
for i, symbol in enumerate(symbols):
    curr_col = get_vol_klines(symbol, starting)
    print(i, len(symbols), len(curr_col))
    cols.append(curr_col)
    time.sleep(0.01)
volume = pd.concat(cols, axis=1)
#down.index = [x.replace(second=0, minute=0, microsecond=0) for x in down.index]
volume

In [ ]:
low_vol = volume.loc['15-Jun-2021':, [x for x in volume.columns if 'DOWNUSDT' not in x]].median(axis=0).sort_values()
low_vol[:100].plot(kind='bar', figsize=(15,5))
low_vol[:20]

In [ ]:
symbols = list(set(to_keep_global+to_keep_down))
starting = int(datetime.datetime.strptime('1-Apr-2021', '%d-%b-%Y').timestamp()*1000)
cols = []
for i, symbol in enumerate(symbols):
    curr_col = get_klines(symbol, starting)
    print(i, len(symbols), len(curr_col))
    cols.append(curr_col)
    time.sleep(0.01)
historical_live_klines = pd.concat(cols, axis=1)
historical_live_klines

In [ ]:
down.iloc[2].iteritems()

In [ ]:
pd.to_datetime(client.get_server_time()['serverTime'], unit='ms')

In [ ]:
with psycopg2.connect('host=localhost user=postgres password=YOUR_DB_PASSWORD port=5432') as conn:
    conn.autocommit = True
    with conn.cursor() as cur:
        for idx, row in down.iterrows():
            print(idx)
            time_str = idx.strftime('%d-%b-%Y %H:%M')
            qry = 'insert into market (time, symbol, close) values(%s, %s, %s)'
            for symbol, val in row.iteritems():
                try:
                    cur.execute(qry, (time_str, symbol, val))
                except UniqueViolation:
                    cur.execute('update market set close=%s where symbol=%s and time=%s', (val, symbol, time_str))


In [ ]:
from psycopg2.errors import UniqueViolation

In [ ]:
sec = get_secret('binance/binance/account2')
clienta = Client(sec['api_key'], sec['api_secret'])

In [ ]:
acc = clienta.get_account()
prices = {x['symbol']: float(x['price']) for x in clienta.get_all_tickers()}

In [ ]:
dust = [x for x in acc['balances'] if x['asset'] not in ['BNB', 'USDT'] and x['asset']+'USDT' in prices and 0. < float(x['free'])*prices[x['asset']+'USDT']<1.]
dust

In [ ]:
for dd in dust:
    try:
        clienta.transfer_dust(asset=dd['asset'])
        time.sleep(0.2)
    except Exception as ex:
        print(ex)

In [ ]:
def get_klines_futures(symbol):
    data = client.futures_klines(symbol=symbol, interval=client.KLINE_INTERVAL_1HOUR, start_str='1-Jan-2021')
    ser = pd.Series([vv[4] for vv in data], dtype=float, index=pd.to_datetime([vv[6] for vv in data], unit='ms'))
    ser.name = symbol
    return ser

In [ ]:
tickers = client.futures_symbol_ticker()
perpetual = []
for i, ticker in enumerate(tickers):
    print(i, len(tickers))
    perpetual.append(get_klines_futures(ticker['symbol']))
    time.sleep(0.1)
perpetual = pd.concat(perpetual, axis=1)
perpetual

In [ ]:
def get_curr_ob_price(symbol, volume, side='bids'):
    ob = client.get_order_book(symbol=symbol)
    data = ob[side]
    cum_amt = 0
    cum_qt = 0
    for _pr, _qt in data:
        pr = float(_pr)
        qt = float(_qt)
        amt = pr*qt
        cum_amt += amt
        cum_qt += qt
        if cum_amt >= volume:
            break
    return cum_amt/cum_qt, float(data[0][0])


In [ ]:
while True:
    x, y = get_curr_ob_price('GTOUSDT', volume=1000)
    print(x, y, f'{(x-y)/x*100:.3f}')

In [ ]:
side = 'asks'
sym = 'DATAUSDT'
pfast, pbest = get_curr_ob_price(sym, volume=1000, side=side)
print(pfast, pbest, f'{(pbest-pfast)/pfast*100:.4f}')
pslow = []
for i in range(10):
    pslow.append(get_curr_ob_price(sym, volume=100, side=side)[0])
    time.sleep(.3)
pslow = sum(pslow)/len(pslow)
print(pfast, pslow, f'{(pslow-pfast)/pfast*100:.4f}')

In [ ]:
def read_account(fld):
    acc = client.get_account()
    tkrs = client.get_orderbook_tickers()
    tkrs = [{'symbol': x['symbol'], 'price': x[fld]} for x in tkrs]
    balances = {x['asset']: float(x['free'])+float(x['locked']) for x in acc['balances']}
    tkrs = {x['symbol']: float(x['price']) for x in tkrs}
    values = {a+'USDT': tkrs.get(a+'USDT', 0)*v for a, v in balances.items()}
    values['USDT'] = balances['USDT']
    return values
aa=read_account('bidPrice')
bb=read_account('askPrice')
print(sum(aa.values()), sum(bb.values()))

In [ ]:
print(client.get_orderbook_tickers()[0])
print(client.get_all_tickers()[0])

In [ ]:
sym = 'GTOUSDT'
pr = round(.05145*0.97, 5)
client.create_order(symbol=sym, side='SELL', type='STOP_LOSS_LIMIT', timeInForce=client.TIME_IN_FORCE_GTC, quantity=15586, price=pr, stopPrice=pr)

In [ ]:
[x for x in infos if x['symbol'] == 'GTOUSDT']

In [ ]:
[x for x in client.get_exchange_info()['symbols'] if x['symbol'] == 'GTOUSDT']

In [ ]:
client.get_account()

In [ ]:
round(.05145*0.97, 5)